# Seleccion de estimadores y sus hiperparámetros

## Score y cross-validated scores

In [2]:
from sklearn import datasets, svm

#cargando los datasets de los digitos que corresponden
# a los números del 0 al 9. Con X las matrices de 8x8
# y la salida de Y es el dígito
X_digits, y_digits = datasets.load_digits(return_X_y=True)

# instancia de la máquina de vector de soporte 
# con un kernel lineal
svc = svm.SVC(C=1, kernel="linear")

# entrenamiento del modelo y calculo de score
# para un modelo de clasificación
svc.fit(
    X_digits[:-100],
    y_digits[:-100],
).score(
    X_digits[-100:],
    y_digits[-100:],
)
# recordemos que el score por defecto es el accuracy
# que representa la proporción de los verdaderos
# positivos y junto con los verdaderos negativos. 
# Por ende, a mayor valor, mejor modelo. Sin embargo,
# esto depepende meramente de la partición de los datos
# que se tiene. Pero hay que buscar una forma de
# seleccionar los mejores hiperparámetros sin importar la
# partición que tenga, por tanto, es que se hace la validadción
# cruzada. 

0.98

In [3]:
# la validación cruzada es un formad de hallar los hiperparámetors
# a traves de ver métricas haciendo diferentes particiones de los datos
# de entrenamiento para selecciona|r los hiperparámetros optimos
# para la generalización del modelo. Sin necesidad de usar los datos 
# de entrenamiento.
# 
# Hay diferentes métodos de cross-validation, uno de los más usados
# es k-fold cross-validation que se basa en tomar los datos de entrenamiento
# y particionarlos en k grupos y con ellos permutaciones de datos 
# de entrenamiento y test par hallar los mejores hiperparámetros.  
import numpy as np
# por ejemplo en este caso se usan 3 folds, usando 2 particiones
# para entrenamiento y la otra partición para test. por ejepmlo, 
# 1 y 2 pronostican 3
# 3 y 1 pronostican 2
# 3 y 2 pronostican 1
# y con esas iteracinoes se busca encontrar los mejores
# hiperparámetros para el modelo. 
X_folds = np.array_split(X_digits, 3)
y_folds = np.array_split(y_digits, 3)

scores = list()
for k in range(3):
    # Se usa'list' para crear una copia, y poder usar 'pop' despué.
    X_train = list(X_folds)
    X_test = X_train.pop(k)
    X_train = np.concatenate(X_train)
    y_train = list(y_folds)
    y_test = y_train.pop(k)
    y_train = np.concatenate(y_train)
    scores.append(svc.fit(X_train, y_train).score(X_test, y_test))
scores

[0.9348914858096828, 0.9565943238731218, 0.9398998330550918]

## Cross-Validated generators


In [4]:
from sklearn.model_selection import KFold, cross_val_score

X = ["a", "a", "a", "b", "b", "c", "c", "c", "c", "c"]

# a continuación puede verse como se hacen las particiones de la validación
# cruzada, en este caso dse hacen 5 particiones con 8 valores de entrenamiento
# y 2 valores de test.
k_fold = KFold(n_splits=5)
for train_indices, test_indices in k_fold.split(X):
    print("Train: %s | test: %s" % (train_indices, test_indices))

Train: [2 3 4 5 6 7 8 9] | test: [0 1]
Train: [0 1 4 5 6 7 8 9] | test: [2 3]
Train: [0 1 2 3 6 7 8 9] | test: [4 5]
Train: [0 1 2 3 4 5 8 9] | test: [6 7]
Train: [0 1 2 3 4 5 6 7] | test: [8 9]


In [5]:
[
    # entrenamos el modelo con las diferentes particiones de los datos
    # además, calculamos los scores con los diferntes datos de test. 
    svc.fit(X_digits[train], y_digits[train]).score(X_digits[test], y_digits[test])
    for train, test in k_fold.split(X_digits)
]

# entonces como se particionarion 5 grupos, que son los que se ven arriba. 
# se entrenan los modelos diferentes y luego se calculan los scores 
# para cada uno de los modelos. Recordemos que esto es para hallar los 
# mejores hiper-parámetros


[0.9638888888888889,
 0.9222222222222223,
 0.9637883008356546,
 0.9637883008356546,
 0.9303621169916435]

In [6]:
# la función de scikit-learn cross_val_score() permite tomar un modelo, 
# el dataset X e Y, el particionamiento y nos devuelve el score
# para cada uno de los grupos. 

#Recordemos que la métrica que se usa para cross validation es 
# el score para modelos de clasificación y varianza explicada 
# para modelos de regresión. 

cross_val_score(
    svc, # modelo
    X_digits, # Regresoras o predictoras
    y_digits, # objetivo
    cv=k_fold,
    n_jobs=-1,
)

array([0.96388889, 0.92222222, 0.9637883 , 0.9637883 , 0.93036212])

In [7]:
# también se puede cambiar la métrica con la se que se evaluan los modelos
# para saber cual es mejor. 
cross_val_score(
    svc,
    X_digits,
    y_digits,
    cv=k_fold,
    scoring="precision_macro",
)

array([0.96578289, 0.92708922, 0.96681476, 0.96362897, 0.93192644])

## Grid-search y cross-validated estimators
GridSearchCV es una herramienta de optimización de hiperparámetros de Scikit-Learn que realiza una búsqueda exhaustiva probando todas las combinaciones posibles de un conjunto de parámetros especificados, evaluando cada combinación mediante validación cruzada (Cross-Validation).

El término combina sus dos pilares: Grid Search (búsqueda en rejilla/malla) + CV (Cross-Validation).


Automatiza el ajuste de hiperparámetros: Evita probar a mano y a ciegas combinaciones de configuración para tu modelo.

Previene el sobreajuste (overfitting): Al usar validación cruzada interna, se asegura de que la mejor combinación de hiperparámetros funcione bien en datos no vistos y no solo en una división particular.

Garantiza objetividad: Encuentra la combinación matemáticamente óptima dentro del espacio de búsqueda definido bajo la métrica que selecciones (como $R^2$, MSE, Accuracy, etc.).

¿Cómo funciona paso a paso?

1. Definición del espacio de búsqueda: Le entregas un diccionario (grid) con los hiperparámetros que quieres probar y los valores para cada uno.

2. Generación del producto cartesiano: Calcula todas las combinaciones posibles.Ejemplo: Si pruebas 3 valores de $C$ y 3 valores de $\gamma$, se generan $3 \times 3 = 9$ combinaciones.

3. Evaluación con $K$-Fold CV: Para cada una de las combinaciones, ejecuta una validación cruzada con $K$ pliegues (por ejemplo, $K=5$).Siguiendo el ejemplo anterior: $9 \text{ combinaciones} \times 5 \text{ folds} = \mathbf{45 \text{ entrenamientos en total}}$

4. Selección del ganador: **Promedia la métrica de evaluación **de los $K$ pliegues para cada combinación y selecciona la que obtenga el mejor rendimiento global.

5. Re-entrenamiento final: Entrena un modelo final utilizando la mejor combinación encontrada sobre la totalidad del conjunto de entrenamiento (X_train, y_train).

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

# 10 valores en escala logaritmica entre 10 ^-6 y 10 ^ -1
Cs = np.logspace(-6, -1, 10)
clf = GridSearchCV(
    estimator=svc, # estimador de máquina de vector de soporte
    param_grid=dict(C=Cs), # en este caso se prueban diferentes Cs
    # y el parámetro debe ser un diccionario o lista de diccionarios
    n_jobs=-1, # número de trabajos a ejecutar en paralelo, 
    # -1 significa ninguno. 
)

# para el entrenamiento, usamos los datos hasta 1000
# tanto en las regresoras como en la varible objetivo
clf.fit(X_digits[:1000], y_digits[:1000])
clf.best_score_ # el mejor score
# que recordemos es el accuracy que representa la tasa
# general de aciertos. 

np.float64(0.95)

In [ ]:
dict(C = Cs) # un diccionario con todos los valores a probar

{'C': array([1.00000000e-06, 3.59381366e-06, 1.29154967e-05, 4.64158883e-05,
        1.66810054e-04, 5.99484250e-04, 2.15443469e-03, 7.74263683e-03,
        2.78255940e-02, 1.00000000e-01])}

In [ ]:
np.logspace(-6, -1, 10)
# retonra los número en una escala logaritmica que por defecto es base 10
# empieza en entonces en este caso en 10 ^-6 y termina en 10^-1.
# El tercer parámetro es la cantidad de valores entre estos. 

array([1.00000000e-06, 3.59381366e-06, 1.29154967e-05, 4.64158883e-05,
       1.66810054e-04, 5.99484250e-04, 2.15443469e-03, 7.74263683e-03,
       2.78255940e-02, 1.00000000e-01])

In [ ]:
clf.best_estimator_.C # Acá miramos el valor óptimo 
# de C que genera ese buen score. Es uno de los del diccionario

np.float64(0.0021544346900318843)

Es importante mencionar que este método por defecto guarda el modelo óptimo y al hacer el método .predict se estaría utilizando el mejor modelo, automáticamente. 

In [ ]:
# prediccion del desempeño sobre el conjunto de test
# como el conjunto de train eran los primero 1000 datos
# para el test serán desde 1000 hasta lo último. 
# Tanto en las predictoras como en la variable objetivo. 
clf.score(
    X_digits[1000:],
    y_digits[1000:],
)

In [ ]:
# nested cross-validation
# particionamiento del mejor modelo obtenido bajo cross-validation
cross_val_score(
    clf,
    X_digits,
    y_digits,
)

array([0.94722222, 0.91666667, 0.96657382, 0.97493036, 0.93593315])

In [ ]:
from sklearn import datasets, linear_model
# hay modelos en sickit learn que ya  tiene implementados estos métodos
# como viene a ser la regresión lasso que ya usa cross validation
# para el uso de su hiperparámetro óptimo. 

# Instancia del modelo con terminación CV pues quiere decir
# que viene incluido con cross validation y prueba valores del 
# alpha que ya vienen predefinidos. 
lasso = linear_model.LassoCV()

# cagamos los datos
X_diabetes, y_diabetes = datasets.load_diabetes(return_X_y=True)

# Hacemos en entrenamiento de los datos. 
lasso.fit(
    X_diabetes,
    y_diabetes,
)

# ver el mejor alpha.
lasso.alpha_

np.float64(0.003753767152691846)